# Curriculum 01 · Lab 4 — HTML Structure

**Goal:** Split a real SEC 10-K filing by its *own* HTML heading hierarchy —
and see why that collapses on XBRL-styled documents.

```
Document       : Apple 10-K — Data/SD-06-tables/aapl-20230930.htm (inline XBRL, ~1.5 MB)
Splitter       : HTMLHeaderTextSplitter (langchain-text-splitters)
Heading chain  : h1 -> H1, h2 -> H2, h3 -> H3 (stored as chunk metadata)
```

HTML documents already carry their own section hierarchy in the heading tags
(`<h1>` … `<h4>`). Instead of splitting on a fixed character/token budget and
hoping a section boundary lands inside a chunk, `HTMLHeaderTextSplitter`
splits *on the headings themselves*: every chunk maps to one document
section, and the heading chain that leads to it is stored as chunk metadata.

    HTML heading hierarchy  ->  chunk metadata  ->  section-scoped retrieval

A retriever can then filter on `metadata["H2"] == "Liquidity and Capital
Resources"` instead of hoping the right text happens to be in the top-k.

## 0 · Setup — dependencies & imports

Three things to know before running:

* **`langchain-text-splitters`** provides `HTMLHeaderTextSplitter` — install cell below.
* **`langchain-core`** provides the `Document` type the splitter returns.
* **No API keys and no network** — parsing is pure local text processing, and
  the sample file is already in the repo (`Data/SD-06-tables/aapl-20230930.htm`).

Run everything from the repo root so the relative data path resolves.

In [1]:
# Needed for THIS lab only:
#   langchain-text-splitters -> HTMLHeaderTextSplitter (split on heading tags)
#   langchain-core           -> Document (the chunk type)
%pip install langchain-text-splitters langchain-core


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

# nbconvert launches the kernel in the notebook's folder, but this lab's data
# path is relative to the repo root (same as the .py). Walk up until the data
# file resolves, so the notebook runs from either location.
for _ in range(6):
    if os.path.exists("Data/SD-06-tables/aapl-20230930.htm"):
        break
    os.chdir("..")

In [3]:
from __future__ import annotations

from collections import Counter

from langchain_core.documents import Document
from langchain_text_splitters import HTMLHeaderTextSplitter

# Module-level constants: the heading chain we split on, and the sample file.
HTML_PATH = "Data/SD-06-tables/aapl-20230930.htm"
HEADERS_TO_SPLIT_ON = [("h1", "H1"), ("h2", "H2"), ("h3", "H3")]
PREVIEW_CHARS = 200


def load_raw_html(path: str) -> str:
    """Read the raw HTML file as text.

    Plain ``open()`` on purpose: the splitter does the parsing, so the lab
    never touches the markup itself.
    """
    with open(path, encoding="utf-8") as f:
        return f.read()


def build_splitter() -> HTMLHeaderTextSplitter:
    """Build the splitter configured for the h1-h3 heading chain."""
    return HTMLHeaderTextSplitter(headers_to_split_on=HEADERS_TO_SPLIT_ON)


def section_counts(chunks: list[Document], level: str) -> Counter[str]:
    """Count chunks per section, keyed by the metadata value for ``level``.

    Chunks with no value for that level (no matching heading tag in the
    document) fall into the ``"(no <tag> found)"`` bucket.
    """
    return Counter(
        chunk.metadata.get(level, f"(no <{level.lower()}> found)") for chunk in chunks
    )


def preview(text: str, limit: int = PREVIEW_CHARS) -> str:
    """Collapse whitespace and cap a content preview at ~``limit`` chars."""
    return " ".join(text.split())[:limit]

## 1 · Load — read the raw SEC 10-K filing

The sample is Apple's 2023 Form 10-K (`aapl-20230930.htm`), a ~1.5 MB
inline-XBRL document. We read it as plain text — the splitter does all the
parsing, so the lab never touches the markup itself.

In [4]:
html = load_raw_html(HTML_PATH)
print(f"[1] Loaded raw HTML from {HTML_PATH} ({len(html):,} bytes)")

[1] Loaded raw HTML from Data/SD-06-tables/aapl-20230930.htm (1,558,924 bytes)


## 2 · Split — on the headings themselves

`HTMLHeaderTextSplitter` walks the document and splits *at* `<h1>`–`<h3>`
tags, carrying the heading chain into each chunk's metadata. On a clean HTML
page that yields one chunk per section. The question this lab asks: what does
a real-world SEC filing — whose headings are bold styled `<span>` elements,
not heading tags — do to that count?

In [5]:
splitter = build_splitter()
chunks = splitter.split_text(html)
print(
    "[2] Split with HTMLHeaderTextSplitter("
    f"headers_to_split_on={HEADERS_TO_SPLIT_ON})"
)
print(f"    -> {len(chunks)} chunk(s)")

[2] Split with HTMLHeaderTextSplitter(headers_to_split_on=[('h1', 'H1'), ('h2', 'H2'), ('h3', 'H3')])
    -> 1 chunk(s)


## 3 · Analyze — what the splitter saw

Chunks that matched no `<h1>`/`<h2>` heading land in a `"(no <tag> found)"`
bucket. We print the distribution across sections, the metadata chain of the
first chunks, and a content preview — the evidence for what actually
happened to the filing's structure.

In [6]:
print("\n[3] Chunk distribution across H1/H2 sections (from chunk metadata):")
for level in ("H1", "H2"):
    counts = section_counts(chunks, level)
    print(f"    metadata[{level!r}]:")
    for section, count in counts.most_common():
        print(f"      {section!r}: {count} chunk(s)")


[3] Chunk distribution across H1/H2 sections (from chunk metadata):
    metadata['H1']:
      '(no <h1> found)': 1 chunk(s)
    metadata['H2']:
      '(no <h2> found)': 1 chunk(s)


In [7]:
print("\n[4] Metadata chain of the first few chunk(s):")
for i, chunk in enumerate(chunks[:3]):
    print(f"    chunk {i}: metadata={chunk.metadata}")


[4] Metadata chain of the first few chunk(s):
    chunk 0: metadata={}


In [8]:
print("\n[5] Content preview (first ~200 chars of chunk 0):")
if chunks:
    print(f"    {preview(chunks[0].page_content)}...")


[5] Content preview (first ~200 chars of chunk 0):
    false 2023 FY 0000320193 P1Y 67 P1Y 25 P1Y 7 1 http://fasb.org/us-gaap/2023#MarketableSecuritiesCurrent http://fasb.org/us-gaap/2023#MarketableSecuritiesNoncurrent http://fasb.org/us-gaap/2023#Marketa...


## 4 · Why one chunk? — XBRL spans, not `<h1>` tags

SEC filings style their headings as bold `<span>` elements
(`font-weight:700`), not `<h1>`–`<h4>` tags — so the splitter finds no
heading chain and collapses the whole filing into a single chunk with empty
metadata. The `<title>` is dropped entirely (the `<head>` is page metadata,
not content), and the styled top heading survives only as plain text inside
the chunk, never as metadata.

In [9]:
print("\n[6] Where did the section headings go? (comparison)")
print(
    "    - <title>aapl-20230930</title>: NOT a header (not in "
    "headers_to_split_on) and its text is dropped from the chunks "
    "entirely — the <head> is page metadata, not content."
)
if chunks:
    content = chunks[0].page_content
    top = content.find("UNITED STATES")
    if top != -1:
        print(
            "    - Top heading 'UNITED STATES / SECURITIES AND EXCHANGE "
            "COMMISSION / FORM 10-K' is a styled <span> (font-weight:700), "
            "NOT an <h1> tag -> it survives as plain text inside the single "
            "chunk but never becomes metadata."
        )
        print(f"      preview around it: {preview(content[top:top + 300])}...")


[6] Where did the section headings go? (comparison)
    - <title>aapl-20230930</title>: NOT a header (not in headers_to_split_on) and its text is dropped from the chunks entirely — the <head> is page metadata, not content.
    - Top heading 'UNITED STATES / SECURITIES AND EXCHANGE COMMISSION / FORM 10-K' is a styled <span> (font-weight:700), NOT an <h1> tag -> it survives as plain text inside the single chunk but never becomes metadata.
      preview around it: UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 10-K (Mark One) ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ...


## 5 · Takeaway

HTML heading hierarchy becomes chunk metadata -> section-scoped retrieval —
but only when the document uses real heading tags. SEC XBRL filings style
headings as spans, so `HTMLHeaderTextSplitter` collapses the whole filing
into one chunk with no metadata; structure-preserving parsing for these
documents needs XBRL/table-aware parsing (the SD-06 track).

In [10]:
print(
    "\n[7] Takeaway: HTML heading hierarchy becomes chunk metadata -> "
    "section-scoped retrieval — but only when the document uses real "
    "heading tags. SEC XBRL filings style headings as spans, so "
    "HTMLHeaderTextSplitter collapses the whole filing into one chunk "
    "with no metadata; structure-preserving parsing for these documents "
    "needs XBRL/table-aware parsing (the SD-06 track)."
)


[7] Takeaway: HTML heading hierarchy becomes chunk metadata -> section-scoped retrieval — but only when the document uses real heading tags. SEC XBRL filings style headings as spans, so HTMLHeaderTextSplitter collapses the whole filing into one chunk with no metadata; structure-preserving parsing for these documents needs XBRL/table-aware parsing (the SD-06 track).
